# Import libs

In [5]:
# import tensorflow and keras
import tensorflow as tf
from tensorflow import keras
from keras import layers
import keras.backend as K

# import numpy to create arrays for trial runs
import numpy as np
# matplotlib for visualisations
import matplotlib.pyplot as plt
# if using Colab, for later saving of models and loading data
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from PIL import Image
from itertools import product
import pandas as pd

Mounted at /content/drive


In [ ]:
#!unzip -q /content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn.zip -d /content/drive/MyDrive/BMET5933/WEEK_10

In [ ]:
# dataset path config
train_vali_dataset_path = Path("/content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn")

# get class names and create mapping to labels
class_names = sorted([folder.name for folder in train_vali_dataset_path.iterdir() if folder.is_dir()])
class_to_label = {class_name: idx for idx, class_name in enumerate(class_names)}
print("Class to label mapping:", class_to_label)


Class to label mapping: {'centromere': 0, 'coarse_speckled': 1, 'fine_speckled': 2, 'homogeneous': 3, 'nucleolar': 4}


AI acknowledge: Chatgpt5.5 used here for learning and suggesting how to design training/validation/test(later) dataset.

In [ ]:
SEED=42
BATCH_SIZE=32
IMAGE_DIR=str(train_vali_dataset_path) # this is the path to the directory containing the class subdirectories
RESCALED_IMAGE_SIZE=(51,51) # images will be all resized to this - they will have 3 channels

training_ds = keras.preprocessing.image_dataset_from_directory(
    IMAGE_DIR,
    batch_size=BATCH_SIZE,
    image_size=RESCALED_IMAGE_SIZE,
		shuffle=True,
		seed = SEED,
    validation_split=0.3,
    subset='training',
		color_mode = 'grayscale'
)

validation_ds = keras.preprocessing.image_dataset_from_directory(
	IMAGE_DIR,
	labels='inferred',
	label_mode='int',
	batch_size=BATCH_SIZE,
	image_size=RESCALED_IMAGE_SIZE,
	shuffle=True,
	seed = SEED,
	validation_split=0.3,
	subset='validation',
	color_mode = 'grayscale'
)

# you can optimise data loading with prefetching
PREFETCH_SIZE = tf.data.AUTOTUNE
training_ds = training_ds.prefetch(buffer_size=PREFETCH_SIZE)
validation_ds = validation_ds.prefetch(buffer_size=PREFETCH_SIZE)


Found 428 files belonging to 5 classes.
Using 300 files for training.
Found 428 files belonging to 5 classes.
Using 128 files for validation.


In [ ]:
# define a shallow CNN model
def shallow_model(input_shape, num_classes, filter_size=5, strides=1, padding='same', num_filters=16):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(num_filters, kernel_size=(filter_size, filter_size), activation='relu', strides=(strides, strides), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(num_classes, activation='softmax'),

	])
	return model

# initialise model and print summary
shallow_cnn = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# check model architecture
shallow_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 51, 51, 16)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        50,005 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,421 (196.96 KB)

 Trainable params: 50,421 (196.96 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# define a deeper CNN
def deeper_model(input_shape, num_classes, filter_size=[3, 3, 3], strides=[1, 1, 1], padding = 'same'):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(32, kernel_size=(filter_size[0], filter_size[0]), activation='relu', strides=(strides[0], strides[0]), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(64, kernel_size=(filter_size[1], filter_size[1]), activation='relu', strides=(strides[1], strides[1]), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(128, kernel_size=(filter_size[2], filter_size[2]), activation='relu', strides=(strides[2], strides[2]), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(64, activation='relu'),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

# initialise deeper model and print summary
deeper_cnn = deeper_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# check deeper model architecture
deeper_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 51, 51, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 25, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 25, 25, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 387,973 (1.48 MB)

 Trainable params: 387,973 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
LEARNING_RATE = 1e-4 # Some common values are 1e-3 (0.001) or 1e-5 (0.00001)
BATCH_SIZE = 32 # train with this many images per iteration [5, 10, or 20 might be good for a trial run]
NUM_EPOCHS = 25 # how many epochs to train for (each epoch visits the training data once) [5 might be good for a trial run]

def train_model(model, training_ds, validation_ds, lr, epochs):
	# model training config
	model.compile(
		optimizer=keras.optimizers.Adam(lr),
		loss="sparse_categorical_crossentropy",
		metrics=["accuracy"]
	)
	training_history = model.fit(
		training_ds,
		validation_data=validation_ds,
		epochs=epochs
	)
	return training_history

# 1. train shallow CNN
# train_history will store the metrics for each epoch, for use in generating graphs
shallow_train = train_model(shallow_cnn, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 86s 9s/step - accuracy: 0.2733 - loss: 8.0573 - val_accuracy: 0.2344 - val_loss: 6.6348
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 187ms/step - accuracy: 0.2633 - loss: 5.0148 - val_accuracy: 0.3125 - val_loss: 4.7445
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - accuracy: 0.3200 - loss: 3.3508 - val_accuracy: 0.3359 - val_loss: 4.0546
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 122ms/step - accuracy: 0.3900 - loss: 2.7251 - val_accuracy: 0.3828 - val_loss: 3.0344
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.3967 - loss: 2.4411 - val_accuracy: 0.3828 - val_loss: 3.1635
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 123ms/step - accuracy: 0.4433 - loss: 2.1070 - val_accuracy: 0.4609 - val_loss: 2.4637
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.4633 - loss: 1.7449 - val_accuracy: 0.3203 - val_loss: 2.8837
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - accuracy: 0.4633 - loss: 2.0466 - val_accuracy: 0.48

In [ ]:
# 2. train deeper CNN
deeper_train = train_model(deeper_cnn, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 7s 411ms/step - accuracy: 0.2067 - loss: 4.8530 - val_accuracy: 0.2109 - val_loss: 2.2232
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 247ms/step - accuracy: 0.2000 - loss: 2.1757 - val_accuracy: 0.3359 - val_loss: 1.4772
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 266ms/step - accuracy: 0.3267 - loss: 1.6261 - val_accuracy: 0.3203 - val_loss: 1.5591
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 240ms/step - accuracy: 0.3967 - loss: 1.5165 - val_accuracy: 0.4766 - val_loss: 1.3781
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 351ms/step - accuracy: 0.4367 - loss: 1.3853 - val_accuracy: 0.6094 - val_loss: 1.1998
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 338ms/step - accuracy: 0.5167 - loss: 1.2365 - val_accuracy: 0.4609 - val_loss: 1.1703
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 247ms/step - accuracy: 0.5533 - loss: 1.1324 - val_accuracy: 0.5547 - val_loss: 1.1181
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 239ms/step - accuracy: 0.5567 - loss: 1.1228 - val_accuracy: 0.

In [ ]:
test_dataset_path = Path('/content/drive/MyDrive/BMET5933/WEEK_10/hep2img_testset')

def read_img(img_path):
  # read image and convert into np array
  img = np.array(Image.open(img_path))

  # expand channel dim
  if img.ndim == 2:
    img = img[..., np.newaxis]
  img = img / 255.0
  img = tf.image.resize(img, (51, 51))
  return img

def model_predict(model, image, class_names: list):
  image = np.array(image)

  if image.ndim == 2:
    # expand channel dim and norm
    image = np.expand_dims(image, axis = 0)
    image = image / 255.0
    image = tf.image.resize(51, 51)

  # predict image label
  prediction = model.predict(image)
  label = np.argmax(prediction, axis = 1)
  return label

def evaluate_model(model, test_ds, class_names: list):
  # get true labels and predicted labels
  true_labels = []
  pred_labels = []
  for image, label in test_ds:
    true_labels.extend(label.numpy())
    pred_label = model_predict(model, image, class_names)
    pred_labels.extend(pred_label)

  # calculate accuracy
  correct = sum(p == t for p, t in zip(pred_labels, true_labels))
  total = len(true_labels)
  accuracy = correct / total
  return accuracy


AI acknowledge: chatgpt5.5 used here for suggesting how to save model.

In [ ]:
# create test dataset
test_ds = keras.preprocessing.image_dataset_from_directory(
	str(test_dataset_path),
	labels='inferred',
	label_mode='int',
	batch_size=BATCH_SIZE,
	image_size=RESCALED_IMAGE_SIZE,
	shuffle=False,
	color_mode = 'grayscale'
)

# evaluate shallow CNN on test set
shallow_accuracy  = evaluate_model(shallow_cnn, test_ds, class_names)
print(f"Shallow CNN Test Accuracy: {shallow_accuracy:.4f}")

#evaluate deeper CNN on test set
deeper_accuracy = evaluate_model(deeper_cnn, test_ds, class_names)
print(f"Deeper CNN Test Accuracy: {deeper_accuracy:.4f}")

# save trained models
shallow_cnn.save('/content/drive/MyDrive/BMET5933/WEEK_10/shallow_cnn.keras')
deeper_cnn.save('/content/drive/MyDrive/BMET5933/WEEK_10/deeper_cnn.keras')
print("Models saved to Google Drive.")


Found 25 files belonging to 5 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Shallow CNN Test Accuracy: 0.3600
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
Deeper CNN Test Accuracy: 1.0000
Models saved to Google Drive.


## Main model comparison

Both CNN models were able to learn from the training data. For the shallow CNN, the final training accuracy was about 0.71 and the final validation accuracy was about 0.57. For the deeper CNN, the final training accuracy was about 0.73 and the final validation accuracy was about 0.71.

On the held-out test set, the shallow CNN got 0.36 accuracy and the deeper CNN got 1.00 accuracy. In my result, the deeper CNN worked much better. I think this is because the deeper CNN has more convolution layers, so it can learn more useful image features from the cell images. However, the test set only has 25 images, so the 1.00 accuracy may change if I use another test split.

# Challenge part:

## 1.replace maxpool with avgpool

In [ ]:
# use avg pooling
def avgpool_shallow_model(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
		layers.AveragePooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(num_classes, activation='softmax'),
	])
	return model

avg_pool_model = avgpool_shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# train avg pooling model
avg_pool_model_training = train_model(avg_pool_model, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

# evaluate average pooling model on test set
avg_pool_accuracy = evaluate_model(avg_pool_model, test_ds, class_names)
print(f"Average Pooling CNN Test Accuracy: {avg_pool_accuracy:.4f}")

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 132ms/step - accuracy: 0.1733 - loss: 8.2721 - val_accuracy: 0.1875 - val_loss: 5.2101
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.2100 - loss: 4.4858 - val_accuracy: 0.1641 - val_loss: 4.2672
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 153ms/step - accuracy: 0.2367 - loss: 3.3531 - val_accuracy: 0.2422 - val_loss: 3.8099
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 148ms/step - accuracy: 0.2600 - loss: 3.2364 - val_accuracy: 0.2656 - val_loss: 3.7724
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 159ms/step - accuracy: 0.2667 - loss: 2.7067 - val_accuracy: 0.3047 - val_loss: 2.9374
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step - accuracy: 0.3367 - loss: 2.2197 - val_accuracy: 0.3125 - val_loss: 2.9502
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.3667 - loss: 2.0778 - val_accuracy: 0.3047 - val_loss: 2.7297
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - accuracy: 0.3867 - loss: 2.0774 - val_accuracy: 0.

### Average pooling conclusion

The average pooling model got 0.56 test accuracy, while the original shallow CNN got 0.36. In this run, average pooling worked better than the original max pooling shallow model. I think average pooling may keep more general brightness and texture information from the cell image regions. However, it was still lower than the deeper CNN result, so it only improved the shallow model in this experiment.

## 2.Parrameter changing
AI acknowledge: chatgpt5.5 used here for giving suggestions on encapsulating function of 'train_model' and polishing explaination.

## Parameter changing note

For this part, I used the original shallow CNN as the baseline. I only changed one convolution parameter each time, so the results are easier to compare:
1. kernel size: 5 -> 7
2. number of filters: 16 -> 32
3. stride: 1 -> 2
4. padding: same -> valid

In [ ]:
# model variations with config
# filter size 7
shallow_model_filter7 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), filter_size=7)
filter7_train = train_model(shallow_model_filter7, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 146ms/step - accuracy: 0.1900 - loss: 7.6549 - val_accuracy: 0.2422 - val_loss: 5.6149
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 162ms/step - accuracy: 0.2067 - loss: 4.8882 - val_accuracy: 0.2500 - val_loss: 4.3636
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 175ms/step - accuracy: 0.2900 - loss: 3.4926 - val_accuracy: 0.2891 - val_loss: 3.0414
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.3300 - loss: 2.6809 - val_accuracy: 0.3203 - val_loss: 2.6439
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 176ms/step - accuracy: 0.3900 - loss: 2.2548 - val_accuracy: 0.4062 - val_loss: 2.6756
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step - accuracy: 0.4333 - loss: 2.0672 - val_accuracy: 0.4062 - val_loss: 2.3698
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 206ms/step - accuracy: 0.4533 - loss: 1.8005 - val_accuracy: 0.4062 - val_loss: 2.2992
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 227ms/step - accuracy: 0.5100 - loss: 1.6234 - val_accuracy: 0.

In [ ]:
# stride 2
shallow_model_stride2 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), strides=2)
stride2_train = train_model(shallow_model_stride2, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - accuracy: 0.1633 - loss: 17.8903 - val_accuracy: 0.2109 - val_loss: 10.1448
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 102ms/step - accuracy: 0.1533 - loss: 8.7861 - val_accuracy: 0.0938 - val_loss: 8.7723
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.1567 - loss: 7.1362 - val_accuracy: 0.2422 - val_loss: 7.4635
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.1400 - loss: 6.3389 - val_accuracy: 0.1250 - val_loss: 7.3332
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.1700 - loss: 5.8349 - val_accuracy: 0.2188 - val_loss: 6.5884
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.1733 - loss: 5.3653 - val_accuracy: 0.1797 - val_loss: 6.2389
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.1500 - loss: 5.1810 - val_accuracy: 0.2031 - val_loss: 5.8251
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.2200 - loss: 4.7210 - val_accuracy: 

In [ ]:
# valid padding
shallow_model_padding_valid = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), padding='valid')
padding_valid_train = train_model(shallow_model_padding_valid, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 140ms/step - accuracy: 0.1367 - loss: 11.1420 - val_accuracy: 0.1641 - val_loss: 9.4721
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 161ms/step - accuracy: 0.1667 - loss: 5.4833 - val_accuracy: 0.1953 - val_loss: 4.5969
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - accuracy: 0.1767 - loss: 3.8064 - val_accuracy: 0.1328 - val_loss: 4.4025
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 173ms/step - accuracy: 0.2133 - loss: 3.1757 - val_accuracy: 0.2500 - val_loss: 3.5995
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.2033 - loss: 2.8825 - val_accuracy: 0.2188 - val_loss: 3.7289
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.2767 - loss: 2.4703 - val_accuracy: 0.1719 - val_loss: 3.5164
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 125ms/step - accuracy: 0.3033 - loss: 2.1980 - val_accuracy: 0.2656 - val_loss: 3.1941
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.2933 - loss: 2.2233 - val_accuracy: 0

In [ ]:
# number of filters 32
shallow_model_filters32 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), num_filters=32)
filters32_train = train_model(shallow_model_filters32, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 144ms/step - accuracy: 0.2000 - loss: 12.8080 - val_accuracy: 0.2422 - val_loss: 6.4159
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 149ms/step - accuracy: 0.2467 - loss: 5.6179 - val_accuracy: 0.1797 - val_loss: 5.7305
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 132ms/step - accuracy: 0.3067 - loss: 4.1700 - val_accuracy: 0.1797 - val_loss: 4.0858
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step - accuracy: 0.3000 - loss: 2.7848 - val_accuracy: 0.3047 - val_loss: 2.7479
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.4133 - loss: 2.3124 - val_accuracy: 0.3125 - val_loss: 2.9756
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 180ms/step - accuracy: 0.3633 - loss: 2.4137 - val_accuracy: 0.3359 - val_loss: 3.2520
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 213ms/step - accuracy: 0.3800 - loss: 2.6231 - val_accuracy: 0.3438 - val_loss: 2.4690
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 187ms/step - accuracy: 0.4733 - loss: 1.9972 - val_accuracy: 0

In [ ]:
# evaluate model variations on test set
filter7_accuracy = evaluate_model(shallow_model_filter7, test_ds, class_names)
print(f"Shallow CNN with filter size 7 Test Accuracy: {filter7_accuracy:.4f}")

filters32_accuracy = evaluate_model(shallow_model_filters32, test_ds, class_names)
print(f"Shallow CNN with 32 filters Test Accuracy: {filters32_accuracy:.4f}")

stride2_accuracy = evaluate_model(shallow_model_stride2, test_ds, class_names)
print(f"Shallow CNN with stride 2 Test Accuracy: {stride2_accuracy:.4f}")

padding_valid_accuracy = evaluate_model(shallow_model_padding_valid, test_ds, class_names)
print(f"Shallow CNN with valid padding Test Accuracy: {padding_valid_accuracy:.4f}")

challenge2_results = pd.DataFrame([
	{"change": "baseline", "filter_size": 5, "num_filters": 16, "stride": 1, "padding": "same", "test_accuracy": shallow_accuracy},
	{"change": "filter size 7", "filter_size": 7, "num_filters": 16, "stride": 1, "padding": "same", "test_accuracy": filter7_accuracy},
	{"change": "32 filters", "filter_size": 5, "num_filters": 32, "stride": 1, "padding": "same", "test_accuracy": filters32_accuracy},
	{"change": "stride 2", "filter_size": 5, "num_filters": 16, "stride": 2, "padding": "same", "test_accuracy": stride2_accuracy},
	{"change": "valid padding", "filter_size": 5, "num_filters": 16, "stride": 1, "padding": "valid", "test_accuracy": padding_valid_accuracy},
])

challenge2_results["accuracy_change"] = challenge2_results["test_accuracy"] - shallow_accuracy
challenge2_results


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
Shallow CNN with filter size 7 Test Accuracy: 0.6400


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
Shallow CNN with 32 filters Test Accuracy: 0.5600


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Shallow CNN with stride 2 Test Accuracy: 0.3200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
Shallow CNN with valid padding Test Accuracy: 0.4400


,change,filter_size,num_filters,stride,padding,test_accuracy,accuracy_change
0,baseline,5,16,1,same,0.36,0.00
1,filter size 7,7,16,1,same,0.64,0.28
2,32 filters,5,32,1,same,0.56,0.20
3,stride 2,5,16,2,same,0.32,-0.04
4,valid padding,5,16,1,valid,0.44,0.08


### Challenge 2 conclusion

From the result table, the 7x7 filter gave the best result with 0.64 test accuracy. The 32-filter model also improved the result to 0.56, and valid padding gave 0.44. In my result, using a larger filter or more filters helped the shallow CNN learn better features. The stride 2 model got 0.32, which was the worst result. I think this is because stride 2 makes the feature map smaller too early, so the model may miss some small cell texture details.

## Challenge 3: Changing training hyperparameters
AI Acknowledge: chatgpt5.5 used in this part for polishing explaination.

In [ ]:
# challenge 3 - change learning rate
# create a new model so the comparison starts from fresh weights
shallow_model_lr_1e3 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# train model with learning rate 1e-3 and original epochs
lr_variation_training = train_model(shallow_model_lr_1e3, training_ds, validation_ds, lr=1e-3, epochs=NUM_EPOCHS)

# evaluate model with learning rate 1e-3
lr_variation_accuracy = evaluate_model(shallow_model_lr_1e3, test_ds, class_names)
print(f"Shallow CNN Test Accuracy with learning rate 1e-3: {lr_variation_accuracy:.4f}")


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 143ms/step - accuracy: 0.1800 - loss: 51.0038 - val_accuracy: 0.1484 - val_loss: 14.6153
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 160ms/step - accuracy: 0.2267 - loss: 12.2241 - val_accuracy: 0.3594 - val_loss: 3.7370
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 158ms/step - accuracy: 0.4200 - loss: 3.6775 - val_accuracy: 0.4531 - val_loss: 2.2917
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.4333 - loss: 1.5205 - val_accuracy: 0.3828 - val_loss: 1.8997
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 136ms/step - accuracy: 0.3567 - loss: 1.4601 - val_accuracy: 0.4766 - val_loss: 1.3404
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 165ms/step - accuracy: 0.5300 - loss: 1.2612 - val_accuracy: 0.4844 - val_loss: 1.3425
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 161ms/step - accuracy: 0.5833 - loss: 1.0741 - val_accuracy: 0.5547 - val_loss: 1.1795
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 219ms/step - accuracy: 0.6667 - loss: 0.9286 - val_accuracy:

In [ ]:
# challenge 3 - change number of epochs
# create another new model so it does not continue from the previous model
shallow_model_epoch15 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# train model with original learning rate and 15 epochs
epoch_variation_training = train_model(shallow_model_epoch15, training_ds, validation_ds, lr=LEARNING_RATE, epochs=15)

# evaluate model with 15 epochs
epoch_variation_accuracy = evaluate_model(shallow_model_epoch15, test_ds, class_names)
print(f"Shallow CNN Test Accuracy with 15 epochs: {epoch_variation_accuracy:.4f}")

challenge3_results = pd.DataFrame([
	{"change": "baseline", "learning_rate": LEARNING_RATE, "epochs": NUM_EPOCHS, "test_accuracy": shallow_accuracy},
	{"change": "learning rate 1e-3", "learning_rate": 1e-3, "epochs": NUM_EPOCHS, "test_accuracy": lr_variation_accuracy},
	{"change": "15 epochs", "learning_rate": LEARNING_RATE, "epochs": 15, "test_accuracy": epoch_variation_accuracy},
])

challenge3_results["accuracy_change"] = challenge3_results["test_accuracy"] - shallow_accuracy
challenge3_results

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 204ms/step - accuracy: 0.1967 - loss: 15.9220 - val_accuracy: 0.1328 - val_loss: 9.2478
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.1733 - loss: 7.8066 - val_accuracy: 0.1094 - val_loss: 6.0183
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.2233 - loss: 5.2253 - val_accuracy: 0.2266 - val_loss: 4.8478
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.2467 - loss: 4.2444 - val_accuracy: 0.2266 - val_loss: 4.3737
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 131ms/step - accuracy: 0.2733 - loss: 4.0751 - val_accuracy: 0.2500 - val_loss: 3.7768
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 116ms/step - accuracy: 0.3300 - loss: 3.1284 - val_accuracy: 0.2500 - val_loss: 3.4777
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.3500 - loss: 2.7743 - val_accuracy: 0.2578 - val_loss: 2.8673
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 175ms/step - accuracy: 0.3933 - loss: 2.3039 - val_accuracy: 0

,change,learning_rate,epochs,test_accuracy,accuracy_change
0,baseline,0.0001,25,0.36,0.00
1,learning rate 1e-3,0.0010,25,0.68,0.32
2,15 epochs,0.0001,15,0.52,0.16


### Challenge 3 conclusion

For the training hyperparameters, learning rate 1e-3 gave the best result in this comparison. It reached 0.68 test accuracy, which was higher than the baseline 0.36. The model trained for 15 epochs also reached 0.52, so it was also better than the baseline in this run. I think the original learning rate 1e-4 may be too slow for the shallow CNN. The results can also change because the test set is small, but in my current result the larger learning rate worked best.

## Challenge 4: Deeper model

In [ ]:
def even_deeper_model(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(32, kernel_size=(5, 5), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(64, kernel_size=(5, 5), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(128, activation='relu'),
		layers.Dense(64, activation='relu'),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

even_deeper_cnn = even_deeper_model(input_shape=(51, 51, 1), num_classes=len(class_names))
even_deeper_cnn.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_11 (Conv2D)              │ (None, 51, 51, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 25, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 25, 25, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 6, 6, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_9 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 724,741 (2.76 MB)

 Trainable params: 724,741 (2.76 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
even_deeper_train = train_model(even_deeper_cnn, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)
even_deeper_accuracy = evaluate_model(even_deeper_cnn, test_ds, class_names)
print(f"Even deeper CNN Test Accuracy: {even_deeper_accuracy:.4f}")

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 408ms/step - accuracy: 0.1967 - loss: 2.8795 - val_accuracy: 0.1953 - val_loss: 1.6555
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 376ms/step - accuracy: 0.2900 - loss: 1.5073 - val_accuracy: 0.2891 - val_loss: 1.4853
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 549ms/step - accuracy: 0.3367 - loss: 1.4294 - val_accuracy: 0.4453 - val_loss: 1.3387
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 403ms/step - accuracy: 0.4633 - loss: 1.3648 - val_accuracy: 0.3438 - val_loss: 1.3687
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 380ms/step - accuracy: 0.4500 - loss: 1.2392 - val_accuracy: 0.5547 - val_loss: 1.2220
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 514ms/step - accuracy: 0.5300 - loss: 1.1798 - val_accuracy: 0.4375 - val_loss: 1.2195
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 449ms/step - accuracy: 0.5633 - loss: 1.1517 - val_accuracy: 0.5312 - val_loss: 1.1374
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 384ms/step - accuracy: 0.5933 - loss: 1.0920 - val_accuracy: 0.

### Challenge 4 conclusion

The even deeper CNN got 0.72 test accuracy. This was better than the shallow CNN result of 0.36, but it was lower than the original deeper CNN result of 1.00. In this run, adding more layers did not make the model better than the deeper CNN. I think the even deeper model may be harder to train well because the dataset is small, which leads to overfitting problem. So my best model in this run was still the original deeper CNN.

In [6]:
# These are for exporting the notebook as a PDF later if needed
#!apt-get update
#!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!jupyter nbconvert --to pdf /content/drive/MyDrive/BMET5933/WEEK_10/Week_10_Li_Code_file_ipynb_final_version.ipynb

[NbConvertApp] Converting notebook /content/drive/MyDrive/BMET5933/WEEK_10/Week_10_Li_Code_file_ipynb_final_version.ipynb to pdf
[NbConvertApp] Writing 143636 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 113052 bytes to /content/drive/MyDrive/BMET5933/WEEK_10/Week_10_Li_Code_file_ipynb_final_version.pdf
